# Python DOCX Text Tests

This notebook demonstrates basic `python-docx` usage and a small budget/receipt generator backed by Pydantic models and JSON mocks.

## 1. Basic Usage: test basic API usage

In [1]:
from pathlib import Path
from docx import Document

In [2]:
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

In [3]:
doc = Document()
doc.add_heading("python-docx basic usage", level=1)
doc.add_paragraph("This paragraph was created from a notebook using python-docx.")

table = doc.add_table(rows=1, cols=3)
table.style = "Table Grid"
for cell, text in zip(table.rows[0].cells, ["Package", "Purpose", "Status"]):
    cell.text = text

row = table.add_row().cells
row[0].text = "python-docx"
row[1].text = "Create and edit Word files"
row[2].text = "Working"

basic_docx = OUTPUT_DIR / "basic_usage.docx"
doc.save(basic_docx)
basic_docx

PosixPath('output/basic_usage.docx')

## 2. Use Case: Quotes

The reusable implementation lives directly in `src/`. JSON inputs are validated with Pydantic, then rendered as Word documents through `Budget` and `Receipt` classes derived from `BaseDocument`. The notebook adds `src/` to `sys.path`, so the modules can be imported without installing this project as a package.

In [4]:
import json
import sys
from pathlib import Path

import pandas as pd

PROJECT_DIR = Path.cwd()
SRC_DIR = PROJECT_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# Import local modules directly from src/. The project itself is not installed.
from documents import Budget, Receipt
from models import BudgetData, ReceiptData

DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "output"
OUTPUT_DIR.mkdir(exist_ok=True)


### Load and Validate JSON Mocks

In [5]:
with (DATA_DIR / "budget.json").open() as f:
    budget_json = json.load(f)

with (DATA_DIR / "receipt.json").open() as f:
    receipt_json = json.load(f)

budget_data = BudgetData.model_validate(budget_json)
receipt_data = ReceiptData.model_validate(receipt_json)

budget_data, receipt_data

(BudgetData(company=CompanyInfo(name='Northwind Robotics SL', tax_id='B12345678', address=Address(street='Calle de la Innovacion 42', postal_code='28020', city='Madrid', country='Spain'), email='billing@northwind-robotics.example', phone='+34 910 000 001', website='https://northwind-robotics.example', logo_path=None), client=ClientInfo(name='Ane', surname='Garcia', tax_id='Y1234567Z', address=Address(street='Paseo de la Concha 10', postal_code='20007', city='Donostia', country='Spain'), email='ane.garcia@example.com', phone='+34 600 000 002'), document_date=datetime.date(2026, 5, 31), iva_rate=Decimal('0.21'), notes='This budget is valid until the date shown above and excludes scope changes requested after approval.', items=[LineItem(description='Automation requirements workshop', unit_price=Decimal('350.00'), quantity=Decimal('1'), unit='day', subtotal=Decimal('350.00')), LineItem(description='Prototype document-generation workflow', unit_price=Decimal('120.00'), quantity=Decimal('6')

### Inspect Line Items

In [6]:
items_df = pd.DataFrame(
    [
        {
            "description": item.description,
            "unit_price": float(item.unit_price),
            "quantity": float(item.quantity),
            "unit": item.unit,
            "subtotal": float(item.subtotal),
        }
        for item in budget_data.items
    ]
)
items_df

,description,unit_price,quantity,unit,subtotal
0,Automation requirements workshop,350.0,1.0,day,350.0
1,Prototype document-generation workflow,120.0,6.0,hour,720.0
2,Deployment and handover session,180.0,1.0,session,180.0


### Generate Budget and Receipt Documents

In [7]:
budget = Budget(budget_data)
receipt = Receipt(receipt_data)

budget_result = budget.export(OUTPUT_DIR, "budget_BUD-2026-0001", pdf=True)
receipt_result = receipt.export(OUTPUT_DIR, "receipt_8b8ddf11", pdf=True)

{
    "budget": {key: str(value) for key, value in budget_result.items()},
    "receipt": {key: str(value) for key, value in receipt_result.items()},
}

  0%|          | 0/1 [00:00<?, ?it/s]

{'input': '/Users/mxagar/nexo/git_repositories/tool_guides/python_docx/output/budget_BUD-2026-0001.docx', 'output': '/Users/mxagar/nexo/git_repositories/tool_guides/python_docx/output/budget_BUD-2026-0001.pdf', 'result': 'error', 'error': 'Error: Message not understood.'}


  0%|          | 0/1 [00:00<?, ?it/s]

{'input': '/Users/mxagar/nexo/git_repositories/tool_guides/python_docx/output/receipt_8b8ddf11.docx', 'output': '/Users/mxagar/nexo/git_repositories/tool_guides/python_docx/output/receipt_8b8ddf11.pdf', 'result': 'error', 'error': 'Error: Message not understood.'}


{'budget': {'docx': '/Users/mxagar/nexo/git_repositories/tool_guides/python_docx/output/budget_BUD-2026-0001.docx',
  'pdf': '/Users/mxagar/nexo/git_repositories/tool_guides/python_docx/output/budget_BUD-2026-0001.pdf'},
 'receipt': {'docx': '/Users/mxagar/nexo/git_repositories/tool_guides/python_docx/output/receipt_8b8ddf11.docx',
  'pdf': '/Users/mxagar/nexo/git_repositories/tool_guides/python_docx/output/receipt_8b8ddf11.pdf'}}

### Totals

In [8]:
totals = pd.DataFrame(
    [
        {
            "document": "budget",
            "net_total": float(budget_data.net_total),
            "iva_total": float(budget_data.iva_total),
            "gross_total": float(budget_data.gross_total),
        },
        {
            "document": "receipt",
            "net_total": float(receipt_data.net_total),
            "iva_total": float(receipt_data.iva_total),
            "gross_total": float(receipt_data.gross_total),
        },
    ]
)
totals

,document,net_total,iva_total,gross_total
0,budget,1250.0,262.5,1512.5
1,receipt,1250.0,262.5,1512.5
